# Test RTE's API

In [ ]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
import requests
from emm.config import CONFIG_PATH, ROOT_PATH

Setup the Open-Meteo API client with cache and retry on error


In [ ]:
locations = yaml.safe_load(CONFIG_PATH.joinpath("locations.yaml").read_text(encoding = "utf-8"))
locations_df = pd.DataFrame(locations["locations"])
latitude = locations_df.loc["lat","ile-de-france"]
longitude = locations_df.loc["lon","ile-de-france"]

Make sure all required weather variables are listed here \
The order of variables in hourly or daily is important to assign them correctly below

In [92]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": latitude,
	"longitude": longitude,
	"hourly": "temperature_2m",
	"minutely_15": "temperature_2m",
}
response = requests.get(url, params = params)
print("Response status code:", response.status_code)
print("Response params:", response.url)
content = response.json()

Response status code: 200
Response params: https://api.open-meteo.com/v1/forecast?latitude=48.863&longitude=2.359&hourly=temperature_2m&minutely_15=temperature_2m


In [76]:
print(f"Coordinates: {content['latitude']}°N {content['longitude']}°E")
print(f"Elevation: {content['elevation']} m asl")
print(f"Timezone difference to GMT+0: {content['utc_offset_seconds']}s")

Coordinates: 48.87072°N 2.3559449°E
Elevation: 48.0 m asl
Timezone difference to GMT+0: 0s


Process minutely_15 data. The order of variables needs to be the same as requested.


In [ ]:
minutely_15_df = pd.DataFrame(content["minutely_15"]).set_index("time")

## Archive API

In [110]:
url = "https://archive-api.open-meteo.com/v1/archive"
latitudes = locations_df.T["lat"].to_numpy()
longitudes = locations_df.T["lon"].to_numpy()
params = {
	"latitude": latitudes,
	"longitude": longitudes,
	"start_date": "2026-10-11",
	"end_date": "2026-10-25",
	"hourly": "temperature_2m",
    "model": "era5_Land",
}
response = requests.get(url, params = params)
print("Response ok: ", response.status_code == 200)
print("Request sent at: ", response.headers.get("Date"))
content = response.json()

Response ok:  True
Request sent at:  Tue, 27 Oct 2026 16:45:06 GMT


In [118]:
hourly_df = pd.concat(
    [
        pd.DataFrame(loc["hourly"]).assign(latitude=loc["latitude"], longitude=loc["longitude"], location_id=i)
        for i,loc in enumerate(content)
    ],
    ignore_index=True,
)
hourly_df["time"] = pd.to_datetime(hourly_df["time"])